In [ ]:
##Advanced QC-Q Index

In [ ]:
#1.Qgaps-Temporal Consistency

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG
# =========================
DATA_DIR = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit"   # โฟลเดอร์ไฟล์รายวัน (เช่น 364 ไฟล์)
ID_COL   = "station_id"
RAIN_COL = "rain(mm)"

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    # รองรับ: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def longest_consecutive_true(mask: pd.Series) -> int:
    """คืนความยาวช่วงต่อเนื่องที่ยาวที่สุดของค่า True ใน Series (เช่นช่องว่างต่อเนื่อง)"""
    if mask.sum() == 0:
        return 0
    grp = (mask != mask.shift()).cumsum()
    return int(mask.groupby(grp).sum().max())

# =========================
# Load all daily files
# =========================
rows = []
file_dates = []
for fp in sorted(Path(DATA_DIR).glob("*.csv")):
    date = parse_date_from_filename(fp)
    file_dates.append(date)
    df = pd.read_csv(fp)

    # มาตรฐานคอลัมน์
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain_mm"})
    # แปลงเป็นตัวเลข; ช่องว่าง/ข้อความ -> NaN
    df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
    # ค่าติดลบ (เป็นไปไม่ได้) -> NaN
    df.loc[df["rain_mm"] < 0, "rain_mm"] = np.nan

    df["date"] = date
    rows.append(df[["station_id", "rain_mm", "date"]])

if not rows:
    raise RuntimeError("No CSV files found.")

df_all = pd.concat(rows, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])
df_all["year"] = df_all["date"].dt.year

# วันทั้งหมดที่ "มีไฟล์จริง"
file_dates = pd.Index(sorted(set(file_dates)))
years_in_files = sorted(pd.unique(file_dates.year))

# หา first/last valid (>=0) ของแต่ละสถานี
bounds = (
    df_all.loc[df_all["rain_mm"].notna()]
         .groupby("station_id")["date"]
         .agg(overall_start="min", overall_end="max")
         .reset_index()
)

all_stations = sorted(df_all["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# เตรียม index ของวันไฟล์รายปี
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# =========================
# Compute Qgaps
# =========================
results = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]   # NaT หากสถานีนี้ไม่เคยมีค่าจริงเลย
    overall_end   = b["overall_end"]
    had_any_valid_ever = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df_all.loc[df_all["station_id"] == sid].copy()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue  # ปีนี้ไม่มีไฟล์เลย

        if had_any_valid_ever:
            # สร้างหน้าต่างปฏิบัติการของปีนี้: [w_start, w_end] = [max(start, Jan1), min(end, Dec31)]
            y_start = pd.Timestamp(year=yr, month=1, day=1)
            y_end   = pd.Timestamp(year=yr, month=12, day=31)
            w_start = max(overall_start, y_start)
            w_end   = min(overall_end, y_end)
            if w_start > w_end:
                continue  # ปีนี้อยู่นอกช่วงปฏิบัติการของสถานี

            # expected_days = จำนวนไฟล์จริงในช่วงนี้
            use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
            n = int(len(use_dates))
            if n == 0:
                # ไม่มีไฟล์ตรงหน้าต่างนี้
                results.append({
                    "station_id": sid, "year": yr,
                    "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                    "expected_days": 0, "n_gap": np.nan, "Lmax_gap": np.nan,
                    "Qgaps": np.nan, "all_year_missing": np.nan,
                    "had_any_valid_ever": True
                })
                continue

            # สร้างซีรีส์รายวันตาม "วันที่มีไฟล์จริงในช่วงนี้" แล้วแมปค่าฝนของสถานี
            s = (g_sta.set_index("date")["rain_mm"]
                      .reindex(use_dates)    # คีย์ตามวันไฟล์จริง
                      .astype(float))

            # missing (NaN) = gap
            gap_mask = s.isna()
            n_gap = int(gap_mask.sum())
            Lmax_gap = longest_consecutive_true(gap_mask)

            # Qgaps
            Qgaps = 100.0 - 100.0 * (2 * n_gap + Lmax_gap) / n
            Qgaps = max(min(Qgaps, 100.0), 0.0)  # clip 0..100

            all_year_missing = (n > 0 and n_gap == n)

            results.append({
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "expected_days": n, "n_gap": n_gap, "Lmax_gap": Lmax_gap,
                "Qgaps": Qgaps, "all_year_missing": all_year_missing,
                "had_any_valid_ever": True
            })

        else:
            # ไม่เคยมีค่าจริงเลยทั้งชุด -> รายงานทุกปีที่มีไฟล์: ทั้งปีเป็น gap
            n = int(len(fdy))
            n_gap = n
            Lmax_gap = n
            Qgaps = 100.0 - 100.0 * (2 * n_gap + Lmax_gap) / n  # = 100 - 300 = -200
            Qgaps = max(min(Qgaps, 100.0), 0.0)  # clip -> 0

            results.append({
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "expected_days": n, "n_gap": n_gap, "Lmax_gap": Lmax_gap,
                "Qgaps": Qgaps, "all_year_missing": True,
                "had_any_valid_ever": False
            })

# =========================
# Save
# =========================
out = pd.DataFrame(results).sort_values(["year", "station_id"])
out_path = Path(DATA_DIR) / "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-1-Qgaps_index.csv"
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(out.head(12))


In [ ]:
#2.P-Data Completeness

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# -----------------------------
# CONFIG
# -----------------------------
DATA_DIR = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit"   # โฟลเดอร์ไฟล์รายวัน
ID_COL   = "station_id"
RAIN_COL = "rain(mm)"

# -----------------------------
# Utils
# -----------------------------
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    # รองรับ 2022-02-03.csv, 20220203.csv, 2022_02_03.csv
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# -----------------------------
# Load all daily files
# -----------------------------
rows = []
file_dates = []
for fp in sorted(Path(DATA_DIR).glob("*.csv")):
    date = parse_date_from_filename(fp)
    file_dates.append(date)
    df = pd.read_csv(fp)

    # มาตรฐานคอลัมน์
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain_mm"})
    # แปลงเป็นตัวเลข; ช่องว่าง/ตัวอักษร -> NaN
    df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
    # ค่าติดลบไม่สมจริง -> NaN
    df.loc[df["rain_mm"] < 0, "rain_mm"] = np.nan

    df["date"] = date
    rows.append(df[["station_id", "rain_mm", "date"]])

if not rows:
    raise RuntimeError("No CSV files found.")

alld = pd.concat(rows, ignore_index=True)
alld["date"] = pd.to_datetime(alld["date"])
alld["year"] = alld["date"].dt.year

# วันทั้งหมดที่ "มีไฟล์จริง" (เช่น 364 วัน)
file_dates = pd.Index(sorted(set(file_dates)))
years_in_files = sorted(pd.unique(file_dates.year))

# หา start/end จาก "วันแรก/สุดท้ายที่สถานีนั้นมีค่าไม่เป็น NaN"
bounds = (
    alld.loc[alld["rain_mm"].notna()]
        .groupby("station_id")["date"]
        .agg(overall_start="min", overall_end="max")
        .reset_index()
)

# สร้างรายการสถานีทั้งหมด (มีชื่อ station_id ปรากฏในไฟล์อย่างน้อย 1 วัน)
all_stations = sorted(alld["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# ทำตารางผลลัพธ์
results = []

# เตรียมดัชนีช่วยนับ expected_days ต่อปี (จากไฟล์จริง)
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]  # อาจเป็น NaT ถ้าไม่เคยมีค่าจริงเลย
    overall_end   = b["overall_end"]

    # มีค่าจริงอย่างน้อย 1 วันในทั้งชุดหรือไม่
    had_any_valid_ever = pd.notna(overall_start) and pd.notna(overall_end)

    # subset ของสถานีนี้ทั้งหมด
    g_sta = alld.loc[alld["station_id"] == sid].copy()

    for yr in years_in_files:
        # expected_days = จำนวน "ไฟล์จริง" ในปีนี้ (จากชื่อไฟล์)
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        expected_days = int(len(fdy))

        if expected_days == 0:
            # ไม่มีไฟล์ของปีนี้เลย -> ข้าม
            continue

        if had_any_valid_ever:
            # จำกัดหน้าต่างปีให้ทับกับช่วงปฏิบัติการจริงของสถานี
            y_start = pd.Timestamp(year=yr, month=1, day=1)
            y_end   = pd.Timestamp(year=yr, month=12, day=31)
            w_start = max(overall_start, y_start)
            w_end   = min(overall_end, y_end)
            if w_start > w_end:
                # ปีนี้อยู่นอกช่วงปฏิบัติการของสถานี
                continue
            # expected_days = นับเฉพาะไฟล์ระหว่าง [w_start, w_end]
            mask_fd = (fdy >= w_start) & (fdy <= w_end)
            exp_days_use = int(mask_fd.sum())

            # นับ n_obs เฉพาะช่วง [w_start, w_end]
            mask_rows = (g_sta["date"] >= w_start) & (g_sta["date"] <= w_end)
            n_obs = int(g_sta.loc[mask_rows, "rain_mm"].notna().sum())

            # ถ้าช่วงปีนี้ทับช่วงปฏิบัติการแต่ไม่มีไฟล์ตรงช่วงเลย
            if exp_days_use == 0:
                # ไม่มีไฟล์ให้เทียบ -> P = NaN
                P = np.nan
            else:
                P = 100.0 * n_obs / exp_days_use

            all_year_missing = (exp_days_use > 0 and n_obs == 0)

            results.append({
                "station_id": sid,
                "year": yr,
                "start_date_used": w_start.date() if pd.notna(w_start) else "",
                "end_date_used": w_end.date() if pd.notna(w_end) else "",
                "expected_days": exp_days_use,
                "n_obs": n_obs,
                "P": P,
                "all_year_missing": all_year_missing,
                "had_any_valid_ever": True
            })

        else:
            # ไม่เคยมีค่าจริงเลยทั้งชุด -> ให้รายงานเป็น P=0 ในทุกปีที่มีไฟล์
            # (ไม่มี start/end ใช้ได้)
            n_obs = 0
            P = 0.0
            results.append({
                "station_id": sid,
                "year": yr,
                "start_date_used": "",
                "end_date_used": "",
                "expected_days": expected_days,
                "n_obs": n_obs,
                "P": P,
                "all_year_missing": True,      # ทั้งปีไม่มีค่าจริง
                "had_any_valid_ever": False    # ทั้ง dataset ไม่มีค่าจริงเลย
            })

# สรุปผล
out = pd.DataFrame(results).sort_values(["year","station_id"])
out_path = Path(DATA_DIR) / "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-2-P_index.csv"
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(out.head(12))


In [ ]:
#3.Qmzero-Descriptive Statistics Review

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG
# =========================
DATA_DIR  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit"   # โฟลเดอร์ไฟล์รายวัน (เช่น 364 ไฟล์)
ID_COL    = "station_id"
DATE_COL  = "date"                   # ถ้าไฟล์มีคอลัมน์ date จะใช้ค่านี้
RAIN_COL  = "rain(mm)"               # ชื่อคอลัมน์ปริมาณฝนตามที่บอก

# พาธบันทึก (พาธเต็ม)
SAVE_Q    = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-3-Qmzero_index.csv"
SAVE_MON  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-3-Qmzero_month_detail.csv"

# นโยบาย Qmzero
COV_THR = 0.90           # monthly coverage threshold (>= 90%)
EXCLUDE_MONTH_END = True # ไม่นับวันสุดท้ายของเดือน

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    # รองรับ 2022-02-03.csv, 20220203.csv, 2022_02_03.csv
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# =========================
# Load all daily files
# =========================
rows = []
for fp in sorted(Path(DATA_DIR).glob("*.csv")):
    df = pd.read_csv(fp)

    # ปรับชื่อคอลัมน์ให้มาตรฐาน
    if ID_COL not in df.columns:
        raise RuntimeError(f"Column '{ID_COL}' not found in {fp.name}")
    df = df.rename(columns={ID_COL: "station_id"})

    if RAIN_COL not in df.columns:
        raise RuntimeError(f"Column '{RAIN_COL}' not found in {fp.name}")
    df = df.rename(columns={RAIN_COL: "rain"})

    # วันที่: ใช้คอลัมน์ date ถ้ามี, ไม่งั้นอ่านจากชื่อไฟล์
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # ทำความสะอาด: แปลงตัวเลข, ค่าติดลบเป็น NaN (ถือว่า missing)
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan

    rows.append(df[["station_id", "rain", "date"]])

if not rows:
    raise RuntimeError("No CSV files found.")

df = pd.concat(rows, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

# >>> ตัดวันสุดท้ายของเดือนออกก่อนคำนวณทั้งหมด <<<
if EXCLUDE_MONTH_END:
    df = df[~df["date"].dt.is_month_end].copy()

df["year"] = df["date"].dt.year

# วันไฟล์จริงทั้งหมด (หลังตัดวันสุดท้ายออกแล้ว)
file_dates = pd.Index(sorted(df["date"].unique()))
years_in_files = sorted(df["year"].unique())

# =========================
# หา first/last valid ของแต่ละสถานี (หลังตัดวันสุดท้ายแล้ว)
# =========================
bounds = (
    df.loc[df["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# เตรียมวันไฟล์จริงแยกปี
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# =========================
# คำนวณ Qmzero + รายละเอียดรายเดือน
# =========================
res_rows = []
mon_rows = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = (
        df.loc[df["station_id"] == sid, ["date", "rain"]]
          .copy()
          .set_index("date")
          .sort_index()
    )

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue  # ปีนี้ไม่มีไฟล์เลย

        if not had_any_valid:
            # ไม่เคยมีค่าจริงเลย → ปีนี้ m=0, m0=0, Qmzero=NaN (ตามนิยาม)
            res_rows.append({"station_id": sid, "year": yr, "m": 0, "m0": 0, "Qmzero": np.nan})
            continue

        # หน้าต่างปีของสถานี
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start) if pd.notna(overall_start) else y_start
        w_end   = min(overall_end, y_end)     if pd.notna(overall_end)   else y_end
        if w_start > w_end:
            # ปีนี้อยู่นอกช่วงปฏิบัติการของสถานี
            continue

        # วันไฟล์จริงภายในหน้าต่างปีนี้
        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            res_rows.append({"station_id": sid, "year": yr, "m": 0, "m0": 0, "Qmzero": np.nan})
            continue

        # ซีรีส์ฝนของสถานีตาม use_dates
        s = g_sta.reindex(use_dates)["rain"].astype(float)

        # ---------- รายเดือนบน "วันไฟล์จริง (ไม่รวมวันสิ้นเดือน)" ----------
        dfm = pd.DataFrame({"rain": s})
        dfm["month"] = dfm.index.to_period("M")

        # operational days = จำนวนวันไฟล์จริงในเดือน (หลังตัดวันสิ้นเดือนแล้ว)
        mon_cnt      = dfm.groupby("month").size()

        # days_with_value = จำนวนวัน non-missing (0 นับเป็นข้อมูล)
        mon_nonnull  = dfm.groupby("month")["rain"].apply(lambda x: x.notna().sum())

        # coverage
        mon_cov = mon_nonnull / mon_cnt

        # eligible month ตามเกณฑ์ ≥ 90%
        eligible = mon_cov >= COV_THR

        # ผลรวมฝนต่อเดือน (เฉพาะวัน non-missing)
        mon_sum = dfm.groupby("month")["rain"].sum(min_count=1)

        # all_zero: มีอย่างน้อย 1 วัน non-missing และค่าสะสมเดือน = 0
        all_zero = (mon_nonnull > 0) & (mon_sum == 0.0)

        # เก็บรายละเอียดรายเดือน (เพื่อดีบัก/ตรวจย้อน)
        mon_detail = pd.DataFrame({
            "station_id": sid,
            "year": yr,
            "month": mon_cnt.index.astype(str),
            "file_days_in_month": mon_cnt.values,         # operational days
            "days_with_value": mon_nonnull.values,
            "coverage": mon_cov.values,
            "eligible": eligible.values,
            "month_sum_mm": mon_sum.values,
            "all_zero": all_zero.values
        })
        mon_rows.append(mon_detail)

        # นับ m และ m0 ตามนิยามใหม่
        m  = int(eligible.sum())
        m0 = int((eligible & all_zero).sum())

        Qmzero = (100.0 - 100.0 * m0 / m) if m > 0 else np.nan
        res_rows.append({"station_id": sid, "year": yr, "m": m, "m0": m0, "Qmzero": Qmzero})

# =========================
# Save outputs
# =========================
res = pd.DataFrame(res_rows).sort_values(["year", "station_id"])
res.to_csv(SAVE_Q, index=False)

if mon_rows:
    mon_df = pd.concat(mon_rows, ignore_index=True)
    mon_df = mon_df.sort_values(["station_id", "year", "month"])
    mon_df.to_csv(SAVE_MON, index=False)

print(f"Saved: {SAVE_Q}")
print(f"Saved: {SAVE_MON}")
print(res.head(10))


In [ ]:
#4.Qwzero-Descriptive Statistics Review

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG
# =========================
DATA_DIR  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit"   # โฟลเดอร์ไฟล์รายวัน (เช่น 364 ไฟล์)
ID_COL    = "station_id"
DATE_COL  = "date"                   # ถ้าไฟล์มีคอลัมน์ date จะใช้ค่านี้
RAIN_COL  = "rain(mm)"               # ชื่อคอลัมน์ปริมาณฝน
RAINY_DAY_THRESHOLD = 1.0            # เกณฑ์ "วันฝน" (มม.)
MIN_RAINY_DAYS      = 20             # ต้องมีวันฝนอย่างน้อยเท่านี้ จึงคำนวณ Qwzero
SAVE_PATH = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-4-Qwzero_index.csv"

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    # รองรับ 2022-02-03.csv, 20220203.csv, 2022_02_03.csv
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# =========================
# Load all daily files
# =========================
rows = []
for fp in sorted(Path(DATA_DIR).glob("*.csv")):
    df = pd.read_csv(fp)

    # ตรวจคอลัมน์พื้นฐาน
    if ID_COL not in df.columns:
        raise RuntimeError(f"Column '{ID_COL}' not found in {fp.name}")
    if RAIN_COL not in df.columns:
        raise RuntimeError(f"Column '{RAIN_COL}' not found in {fp.name}")

    # รีเนมให้ง่าย
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})

    # วันที่: ใช้ในไฟล์ถ้ามี ไม่งั้นอ่านจากชื่อไฟล์
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # ทำความสะอาดค่า
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan  # ค่าติดลบ => missing

    rows.append(df[["station_id", "rain", "date"]])

if not rows:
    raise RuntimeError("No CSV files found.")

df = pd.concat(rows, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

# วันไฟล์จริงทั้งหมด และรายปี
file_dates = pd.Index(sorted(df["date"].unique()))
years_in_files = sorted(df["year"].unique())
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# หาวันแรก/วันสุดท้ายที่มีค่าจริง (≥0) ของแต่ละสถานี
bounds = (
    df.loc[df["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# =========================
# Compute Qwzero
# =========================
results = []
dow_names = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df.loc[df["station_id"] == sid, ["date", "rain"]].copy().set_index("date").sort_index()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue  # ปีนี้ไม่มีไฟล์เลย

        if not had_any_valid:
            # ไม่เคยมีค่าจริงทั้งชุด -> ปีนี้คำนวณไม่ได้
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "rainy_days_total": 0, "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "Qwzero": np.nan
            }
            # เติมคอลัมน์นับต่อวันในสัปดาห์ด้วย 0
            for i in range(7): out[f"rainy_{dow_names[i]}"] = 0
            results.append(out)
            continue

        # ตัดหน้าต่างปีของสถานี
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start)
        w_end   = min(overall_end, y_end)
        if w_start > w_end:
            continue

        # วันไฟล์จริงภายในหน้าต่างปีนี้
        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "rainy_days_total": 0, "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "Qwzero": np.nan
            }
            for i in range(7): out[f"rainy_{dow_names[i]}"] = 0
            results.append(out)
            continue

        # ซีรีส์ของสถานีตามวันไฟล์จริง
        s = g_sta.reindex(use_dates)["rain"].astype(float)

        # นิยาม "วันฝน"
        rainy = (s >= RAINY_DAY_THRESHOLD)
        rainy_days_total = int(rainy.sum())

        # เงื่อนไขขั้นต่ำ
        if rainy_days_total < MIN_RAINY_DAYS:
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "rainy_days_total": rainy_days_total,
                "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "Qwzero": np.nan
            }
            counts = pd.Series(0, index=range(7))
            if rainy_days_total > 0:
                # นับตามวันในสัปดาห์
                dows = pd.Index(use_dates).weekday
                counts = pd.Series(rainy.astype(int).values, index=dows).groupby(level=0).sum().reindex(range(7), fill_value=0)
            for i in range(7): out[f"rainy_{dow_names[i]}"] = int(counts[i])
            results.append(out)
            continue

        # นับวันฝนแยกตามวันในสัปดาห์
        dows = pd.Index(use_dates).weekday  # Monday=0 ... Sunday=6
        counts = pd.Series(rainy.astype(int).values, index=dows).groupby(level=0).sum().reindex(range(7), fill_value=0)

        mean_ = float(counts.mean())
        std_  = float(counts.std(ddof=0))
        cv_   = (std_ / mean_) if mean_ > 0 else np.nan
        Qwzero = 100.0 - 100.0 * cv_ if pd.notna(cv_) else np.nan
        if pd.notna(Qwzero):
            Qwzero = max(min(Qwzero, 100.0), 0.0)

        out = {
            "station_id": sid, "year": yr,
            "start_date_used": w_start.date(), "end_date_used": w_end.date(),
            "rainy_days_total": rainy_days_total,
            "mean_by_dow": mean_, "std_by_dow": std_, "cv_by_dow": cv_, "Qwzero": Qwzero
        }
        for i in range(7): out[f"rainy_{dow_names[i]}"] = int(counts[i])
        results.append(out)

# =========================
# Save
# =========================
res = pd.DataFrame(results).sort_values(["year", "station_id"])
res.to_csv(Path(DATA_DIR) / SAVE_PATH, index=False)
print(f"Saved: {Path(DATA_DIR) / SAVE_PATH}")
print(res.head(12))


In [ ]:
#5.Qoutliers-Outlier Detection

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# =========================
# CONFIG
# =========================
DATA_DIR  = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/1-3_Thaiwater_Physical-Limit"   # โฟลเดอร์ไฟล์รายวัน
ID_COL    = "station_id"
DATE_COL  = "date"                   # ถ้าไม่มีคอลัมน์นี้ จะอ่านจากชื่อไฟล์แทน
RAIN_COL  = "rain(mm)"               # คอลัมน์ปริมาณฝน
RAINY_DAY_THRESHOLD = 1.0            # วันฝน = ค่าฝน ≥ 1 มม.
SAVE_YR   = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-5-Qoutliers_index.csv"
SAVE_MON  = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-5-Qoutliers_month_detail.csv"

# =========================
# Utils
# =========================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    # รองรับชื่อไฟล์ 2022-02-03.csv / 20220203.csv / 2022_02_03.csv
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# =========================
# Load all daily files
# =========================
rows = []
for fp in sorted(Path(DATA_DIR).glob("*.csv")):
    df = pd.read_csv(fp)

    # ตรวจคอลัมน์
    if ID_COL not in df.columns:
        raise RuntimeError(f"Column '{ID_COL}' not found in {fp.name}")
    if RAIN_COL not in df.columns:
        raise RuntimeError(f"Column '{RAIN_COL}' not found in {fp.name}")

    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})

    # วันที่: ใช้คอลัมน์ในไฟล์ถ้ามี ไม่งั้นอ่านจากชื่อไฟล์
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # ทำความสะอาดค่า
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan  # ค่าติดลบ => missing

    rows.append(df[["station_id", "rain", "date"]])

if not rows:
    raise RuntimeError("No CSV files found.")

df = pd.concat(rows, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

# วันไฟล์จริง (ใช้จำกัดช่วง/ตัวหาร)
file_dates = pd.Index(sorted(df["date"].unique()))
years_in_files = sorted(df["year"].unique())
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# ช่วงปฏิบัติการของแต่ละสถานี (วันแรก/สุดท้ายที่มีค่าไม่เป็น NaN)
bounds = (
    df.loc[df["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# =========================
# Qoutliers per station-year
# =========================
yr_rows = []
mon_rows = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df.loc[df["station_id"] == sid, ["date", "rain"]].copy().set_index("date").sort_index()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue  # ปีนี้ไม่มีไฟล์เลย

        if not had_any_valid:
            # ไม่มีข้อมูลจริงทั้งชุด -> ปีนี้ N_obs = 0 -> Qoutliers = NaN
            yr_rows.append({
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "N_obs": 0, "N_out": 0, "Qoutliers": np.nan
            })
            continue

        # ตัดหน้าต่างปีของสถานี
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start)
        w_end   = min(overall_end, y_end)
        if w_start > w_end:
            continue

        # วันไฟล์จริงในหน้าต่างปีนี้
        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            yr_rows.append({
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "N_obs": 0, "N_out": 0, "Qoutliers": np.nan
            })
            continue

        # ซีรีส์รายวันของสถานีตามวันไฟล์จริง
        s = g_sta.reindex(use_dates)["rain"].astype(float)
        N_obs = int(s.notna().sum())

        # วนรายเดือนบนวันไฟล์จริง
        monthly = s.to_frame("rain")
        monthly["month"] = monthly.index.to_period("M")

        N_out = 0
        month_details = []

        for mth, sub in monthly.groupby("month"):
            vals = sub["rain"].dropna()
            obs_days = int(vals.size)             # วันมีข้อมูลในเดือน
            rainy_vals = vals[vals >= RAINY_DAY_THRESHOLD]  # วันฝน (≥1 มม.)

            if rainy_vals.size == 0:
                # ไม่มีวันฝนในเดือนนี้ -> สร้าง T ไม่ได้ -> ไม่มี outlier
                q1 = q3 = iqr = thr = np.nan
                out_days = 0
            else:
                q1 = float(rainy_vals.quantile(0.25))
                q3 = float(rainy_vals.quantile(0.75))
                iqr = q3 - q1
                thr = q3 + 3.0 * iqr
                out_days = int((vals > thr).sum())

            N_out += out_days

            month_details.append({
                "station_id": sid,
                "year": yr,
                "month": str(mth),
                "obs_days_in_month": obs_days,
                "rainy_days_in_month": int(rainy_vals.size),
                "Q1": q1, "Q3": q3, "IQR": iqr, "T": thr,
                "outlier_days_in_month": out_days
            })

        # คำนวณ Qoutliers
        if N_obs > 0:
            Qoutliers = 100.0 * (1.0 - N_out / N_obs)
            Qoutliers = max(min(Qoutliers, 100.0), 0.0)
        else:
            Qoutliers = np.nan

        yr_rows.append({
            "station_id": sid, "year": yr,
            "start_date_used": w_start.date(), "end_date_used": w_end.date(),
            "N_obs": N_obs, "N_out": N_out, "Qoutliers": Qoutliers
        })
        if month_details:
            mon_rows.extend(month_details)

# =========================
# Save outputs
# =========================
yr_df = pd.DataFrame(yr_rows).sort_values(["year", "station_id"])
yr_df.to_csv(Path(DATA_DIR) / SAVE_YR, index=False)

if mon_rows:
    mon_df = pd.DataFrame(mon_rows).sort_values(["station_id", "year", "month"])
    mon_df.to_csv(Path(DATA_DIR) / SAVE_MON, index=False)

print(f"Saved: {Path(DATA_DIR) / SAVE_YR}")
print(f"Saved: {Path(DATA_DIR) / SAVE_MON}")
print(yr_df.head(10))


In [ ]:
#6.Q-Index

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ======================
# CONFIG
# ======================
EXCEL_PATH = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2024/2-6-Q_index.xlsx"  # ไฟล์ Excel ตามภาพ
SHEET_NAME = 0  # 0 = แผ่นงานแรก; หรือใส่ชื่อชีต เช่น "Sheet1"
ID_COLS = ["station_id"]  # คอลัมน์คีย์ (ใส่เพิ่มได้ เช่น ["station_id","year"])
TERMS = ["P", "Qgaps", "Qmzero", "Qwzero", "Qoutliers"]
OUT_PATH = None  # ถ้า None จะเซฟไฟล์ใหม่ข้างๆเป็น *_with_Q.xlsx

# ======================
# Load
# ======================
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

# ให้แน่ใจว่าคอลัมน์ดัชนีอยู่ครบ (ถ้าชื่อสะกดต่าง ให้แก้ใน TERMS ให้ตรง)
missing_cols = [c for c in TERMS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# แปลงดัชนีเป็นตัวเลข; สิ่งที่ไม่ใช่ตัวเลขจะกลายเป็น NaN
df[TERMS] = df[TERMS].apply(pd.to_numeric, errors="coerce")

# Clip ดัชนีให้อยู่ในช่วง 0..100 เผื่อมีค่าผิดพลาด
df[TERMS] = df[TERMS].clip(lower=0, upper=100)

# ======================
# Compute Q
# ======================
# จำนวนดัชนีที่มีค่า (ไม่นับ NaN)
df["terms_used"] = df[TERMS].notna().sum(axis=1)

# Q แบบ "หารตามจำนวนที่มี" (นับ 0 เป็นค่าได้, ไม่นับ NaN)
# ถ้าไม่มีดัชนีไหนเลย (terms_used=0) จะให้เป็น NaN
sum_terms = df[TERMS].sum(axis=1, skipna=True)
df["Q"] = sum_terms / df["terms_used"].replace({0: np.nan})

# (ทางเลือก) Q เฉพาะแถวที่มีอย่างน้อย 3 ดัชนี
df["Q_min3"] = np.where(df["terms_used"] >= 3, df["Q"], np.nan)

# (ทางเลือก) Q ต้องครบทั้ง 5 ดัชนี
df["Q_strict5"] = df[TERMS].mean(axis=1, skipna=False)  # ถ้าขาดสักตัวจะเป็น NaN

# ปัดทศนิยมตามต้องการ
df[["Q", "Q_min3", "Q_strict5"]] = df[["Q", "Q_min3", "Q_strict5"]].round(3)

# ======================
# Save
# ======================
if OUT_PATH is None:
    p = Path(EXCEL_PATH)
    OUT_PATH = str(p.with_name(p.stem + "_with_Q.xlsx"))

# จัดลำดับคอลัมน์ให้อ่านง่าย
front_cols = [c for c in ID_COLS if c in df.columns] + TERMS + ["terms_used", "Q", "Q_min3", "Q_strict5"]
other_cols = [c for c in df.columns if c not in front_cols]
df = df[front_cols + other_cols]

df.to_excel(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")


In [ ]:
#7.Q-Index Cut-off

In [ ]:
import pandas as pd
from pathlib import Path

# =======================
# CONFIG (แก้ให้ตรงของคุณ)
# =======================
EXCEL_PATH = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2022/2-6-Q_index_with_Q.xlsx"   # ไฟล์รวม Q
CSV_DIR    = r"D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2022/1-4_Thaiwater_Repeated"                # โฟลเดอร์ไฟล์รายวัน 364 ไฟล์
ID_COL     = "station_id"                          # ชื่อคอลัมน์รหัสสถานีใน CSV
Q_COL      = "Q"                                   
OUT_DIR    = "D:/PhD-Semester-2/Data-preprocessing/Input/TMD-2022/2-6-Q_index"   # ถ้า None จะสร้างโฟลเดอร์ย่อยชื่อ "cleaned" ใน CSV_DIR
DRY_RUN    = False  # True = แค่พิมพ์จำนวนที่จะลบ แต่ไม่เซฟไฟล์

# ถ้าสตรีง station_id มี '.0' พ่วงมาจาก Excel โค้ดนี้จะจัดรูปให้ตรงกัน
def normalize_id_series(s: pd.Series) -> pd.Series:
    # แปลงเป็นสตริง ตัดช่องว่าง และตัดท้าย ".0"
    return (s.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True))

# ==============
# Load Excel
# ==============
dfq = pd.read_excel(EXCEL_PATH)
if ID_COL not in dfq.columns or Q_COL not in dfq.columns:
    raise ValueError(f"Excel must contain columns '{ID_COL}' and '{Q_COL}'.")

# หา station_id ที่ Q < 50
low_q_ids = normalize_id_series(dfq.loc[pd.to_numeric(dfq[Q_COL], errors="coerce") < 50, ID_COL]).unique()
low_q_ids = set([x for x in low_q_ids if x != "nan"])  # กัน NaN ที่กลายเป็นสตริง

print(f"Low-Q stations (Q < 50): {len(low_q_ids)} found")

# ==============
# Prepare output
# ==============
csv_dir = Path(CSV_DIR)
if OUT_DIR is None:
    out_dir = csv_dir / "cleaned"
else:
    out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# ==============
# Process CSVs
# ==============
total_removed = 0
total_rows_before = 0
files = sorted(csv_dir.glob("*.csv"))
if not files:
    raise RuntimeError(f"No CSV files in {CSV_DIR}")

for fp in files:
    df = pd.read_csv(fp)
    if ID_COL not in df.columns:
        raise ValueError(f"{fp.name} does not contain column '{ID_COL}'")

    # คอลัมน์ช่วยสำหรับเทียบรหัส
    df["_id_norm"] = normalize_id_series(df[ID_COL])
    rows_before = len(df)

    # ลบสถานีที่ Q < 50
    mask_remove = df["_id_norm"].isin(low_q_ids)
    removed = int(mask_remove.sum())
    kept_df = df.loc[~mask_remove].drop(columns=["_id_norm"])

    total_removed += removed
    total_rows_before += rows_before

    if DRY_RUN:
        print(f"[DRY] {fp.name}: removed {removed} rows, kept {len(kept_df)}")
    else:
        kept_df.to_csv(out_dir / fp.name, index=False)
        print(f"{fp.name}: removed {removed} rows, kept {len(kept_df)}")

print(f"\nSummary: removed {total_removed} / {total_rows_before} rows across {len(files)} files.")
print(f"Output folder: {out_dir}")
